# Bearing Fault Dataset Analysis

Overview of bearing fault datasets for Mechanical-JEPA experiments.

**Datasets**: CWRU (classification benchmark) + IMS (run-to-failure)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy import signal
import json
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data/bearings')

## 1. Dataset Overview

In [ ]:
# Load processed metadata
df = pd.read_parquet(DATA_DIR / 'bearing_episodes.parquet')
with open(DATA_DIR / 'statistics.json') as f:
    stats = json.load(f)

print("=" * 60)
print("BEARING FAULT DATASETS SUMMARY")
print("=" * 60)

for ds_name, ds_stats in stats['by_dataset'].items():
    duration_min = ds_stats['n_samples_total'] / ds_stats['sampling_rate'] / 60
    print(f"\n{ds_name.upper()}:")
    print(f"  Episodes:      {ds_stats['n_episodes']:,}")
    print(f"  Total samples: {ds_stats['n_samples_total']:,}")
    print(f"  Duration:      {duration_min:.1f} minutes")
    print(f"  Channels:      {ds_stats['n_channels']}")
    print(f"  Sampling rate: {ds_stats['sampling_rate']/1000:.0f} kHz")
    print(f"  Fault types:   {ds_stats['fault_types']}")

print("\n" + "=" * 60)
print(f"TOTAL: {stats['n_episodes']:,} episodes")
print("=" * 60)

In [ ]:
# Visual summary
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Episodes per dataset
ax = axes[0]
ds_episodes = df.groupby('dataset').size()
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
bars = ax.bar(ds_episodes.index, ds_episodes.values, color=colors[:len(ds_episodes)])
ax.set_ylabel('Episodes')
ax.set_title('Episodes per Dataset')
ax.set_yscale('log')
for bar, val in zip(bars, ds_episodes.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1, 
            f'{val:,}', ha='center', fontsize=10)

# 2. Fault type distribution
ax = axes[1]
fault_counts = df['fault_type'].value_counts()
fault_colors = {'healthy': '#2ecc71', 'inner_race': '#e74c3c', 
                'ball': '#3498db', 'outer_race': '#f39c12'}
bars = ax.bar(fault_counts.index, fault_counts.values, 
              color=[fault_colors.get(f, '#95a5a6') for f in fault_counts.index])
ax.set_ylabel('Episodes')
ax.set_title('Fault Type Distribution')
ax.tick_params(axis='x', rotation=20)
ax.set_yscale('log')

# 3. Samples per dataset (data volume)
ax = axes[2]
ds_samples = df.groupby('dataset')['n_samples'].sum() / 1e6
bars = ax.bar(ds_samples.index, ds_samples.values, color=colors[:len(ds_samples)])
ax.set_ylabel('Samples (millions)')
ax.set_title('Data Volume per Dataset')
for bar, val in zip(bars, ds_samples.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
            f'{val:.0f}M', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 2. CWRU Dataset: Classification Benchmark

4 fault types × 3 severity levels × 4 load conditions

In [ ]:
cwru = df[df['dataset'] == 'cwru'].copy()

# Load actual signals
def load_cwru_signal(file_id):
    mat_file = DATA_DIR / 'raw' / 'cwru' / f'{file_id}.mat'
    if not mat_file.exists():
        return None, None
    data = loadmat(str(mat_file), squeeze_me=True)
    # Find DE (drive end) channel
    for key in data.keys():
        if '_DE_time' in key:
            return data[key], 12000
    return None, None

# Load one example from each fault type
examples = {}
for fault in ['healthy', 'inner_race', 'ball', 'outer_race']:
    file_id = cwru[cwru['fault_type'] == fault]['bearing_id'].iloc[0]
    sig, fs = load_cwru_signal(file_id)
    if sig is not None:
        examples[fault] = (sig, fs, file_id)

print(f"Loaded {len(examples)} fault type examples from CWRU")

In [ ]:
# Time domain comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes = axes.flatten()

n_show = 4096  # ~0.34 seconds
fault_colors = {'healthy': '#2ecc71', 'inner_race': '#e74c3c', 
                'ball': '#3498db', 'outer_race': '#f39c12'}

for ax, (fault, (sig, fs, file_id)) in zip(axes, examples.items()):
    t = np.arange(n_show) / fs * 1000  # ms
    ax.plot(t, sig[:n_show], linewidth=0.5, color=fault_colors[fault], alpha=0.8)
    ax.set_title(f'{fault.replace("_", " ").title()} ({file_id})', fontweight='bold')
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, t[-1])
    
    # Add RMS annotation
    rms = np.sqrt(np.mean(sig[:n_show]**2))
    ax.text(0.98, 0.95, f'RMS: {rms:.4f}', transform=ax.transAxes, 
            ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[-2].set_xlabel('Time (ms)')
axes[-1].set_xlabel('Time (ms)')
fig.suptitle('CWRU: Time Domain Signals by Fault Type (Drive End Accelerometer)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Frequency domain comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes = axes.flatten()

for ax, (fault, (sig, fs, file_id)) in zip(axes, examples.items()):
    # Welch PSD
    freqs, psd = signal.welch(sig, fs=fs, nperseg=2048)
    ax.semilogy(freqs, psd, linewidth=0.8, color=fault_colors[fault])
    ax.set_title(f'{fault.replace("_", " ").title()}', fontweight='bold')
    ax.set_ylabel('PSD')
    ax.set_xlim(0, 5000)
    ax.grid(True, alpha=0.3)

axes[-2].set_xlabel('Frequency (Hz)')
axes[-1].set_xlabel('Frequency (Hz)')
fig.suptitle('CWRU: Power Spectral Density by Fault Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Spectrogram comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes = axes.flatten()

for ax, (fault, (sig, fs, file_id)) in zip(axes, examples.items()):
    f, t, Sxx = signal.spectrogram(sig[:fs*2], fs=fs, nperseg=256, noverlap=128)
    ax.pcolormesh(t*1000, f, 10*np.log10(Sxx + 1e-10), shading='gouraud', cmap='viridis')
    ax.set_title(f'{fault.replace("_", " ").title()}', fontweight='bold')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_ylim(0, 5000)

axes[-2].set_xlabel('Time (ms)')
axes[-1].set_xlabel('Time (ms)')
fig.suptitle('CWRU: Spectrograms by Fault Type (2 seconds)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. IMS Dataset: Run-to-Failure

Continuous monitoring until bearing failure (~weeks of operation)

In [ ]:
ims = df[df['dataset'] == 'ims'].copy()

# Load example IMS signal
ims_dir = DATA_DIR / 'raw' / 'ims' / '1st_test'
ims_files = sorted([f for f in ims_dir.iterdir() if f.is_file() and f.name[0].isdigit()])[:5]

def load_ims_signal(filepath):
    try:
        data = np.loadtxt(filepath)
        return data, 20000
    except:
        return None, None

print(f"IMS 1st_test: {len(list(ims_dir.iterdir()))} files")
print(f"Loading first 5 files for visualization...")

In [ ]:
# Load early vs late measurements (degradation)
all_ims_files = sorted([f for f in ims_dir.iterdir() if f.is_file() and f.name[0].isdigit()])
early_file = all_ims_files[0]
late_file = all_ims_files[-1]

early_sig, fs = load_ims_signal(early_file)
late_sig, _ = load_ims_signal(late_file)

if early_sig is not None and late_sig is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 6))
    
    n_show = 4096
    t = np.arange(n_show) / fs * 1000
    
    # Time domain
    axes[0, 0].plot(t, early_sig[:n_show, 0], linewidth=0.5, color='#2ecc71')
    axes[0, 0].set_title(f'Early: {early_file.name} (Bearing 1)', fontweight='bold')
    axes[0, 0].set_ylabel('Amplitude')
    
    axes[0, 1].plot(t, late_sig[:n_show, 0], linewidth=0.5, color='#e74c3c')
    axes[0, 1].set_title(f'Late: {late_file.name} (Bearing 1)', fontweight='bold')
    
    # Frequency domain
    freqs, psd_early = signal.welch(early_sig[:, 0], fs=fs, nperseg=2048)
    freqs, psd_late = signal.welch(late_sig[:, 0], fs=fs, nperseg=2048)
    
    axes[1, 0].semilogy(freqs, psd_early, linewidth=0.8, color='#2ecc71')
    axes[1, 0].set_xlabel('Frequency (Hz)')
    axes[1, 0].set_ylabel('PSD')
    axes[1, 0].set_xlim(0, 8000)
    
    axes[1, 1].semilogy(freqs, psd_late, linewidth=0.8, color='#e74c3c')
    axes[1, 1].set_xlabel('Frequency (Hz)')
    axes[1, 1].set_xlim(0, 8000)
    
    fig.suptitle('IMS: Early vs Late Measurements (Degradation Over Time)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # RMS trend
    print(f"\nRMS comparison (Bearing 1):")
    print(f"  Early: {np.sqrt(np.mean(early_sig[:, 0]**2)):.4f}")
    print(f"  Late:  {np.sqrt(np.mean(late_sig[:, 0]**2)):.4f}")
    print(f"  Ratio: {np.sqrt(np.mean(late_sig[:, 0]**2)) / np.sqrt(np.mean(early_sig[:, 0]**2)):.2f}x")

## 4. Key Observations for JEPA

### What the data shows:
1. **Fault signatures are in frequency domain** - Different fault types have distinct spectral patterns
2. **Temporal structure matters** - Spectrograms show time-varying patterns
3. **Degradation is progressive** - IMS shows gradual amplitude increase

### Implications for Mechanical-JEPA:
- **Patch size**: Should capture multiple shaft rotations (~10-50ms)
- **Masking**: Temporal blocks make sense; physics-aware masking for Paderborn
- **Pretext task**: Predicting masked patches should learn fault-relevant features

### Scale comparison with Brain-JEPA:

| Metric | Brain-JEPA | Bearings (Current) |
|--------|------------|-------------------|
| Independent samples | 40,000 subjects | ~90 bearings |
| Total data points | ~2.9B | ~200M |
| Sampling rate | 0.5 Hz | 12-20 kHz |
| Spatial features | 450 ROIs | 2-8 channels |

In [ ]:
# Final summary statistics
print("\n" + "="*60)
print("FINAL DATASET SUMMARY FOR MECHANICAL-JEPA")
print("="*60)

total_samples = df['n_samples'].sum()
total_duration_hr = sum(
    stats['by_dataset'][ds]['n_samples_total'] / stats['by_dataset'][ds]['sampling_rate']
    for ds in stats['by_dataset']
) / 3600

print(f"\nTotal episodes:     {len(df):,}")
print(f"Total samples:      {total_samples:,}")
print(f"Total duration:     {total_duration_hr:.1f} hours")
print(f"Datasets:           {list(stats['by_dataset'].keys())}")
print(f"\nFault distribution:")
for fault, count in stats['fault_distribution'].items():
    pct = count / len(df) * 100
    print(f"  {fault:12s}: {count:6,} ({pct:5.1f}%)")